# Leftover Lens: first segmentation model (Colab, T4 GPU)

Before running:
1. Runtime > Change runtime type > T4 GPU.
2. Upload `leftover-lens.zip` to the top level of your Google Drive (My Drive).

Voxel51's labels cover only part of our class list. Read the summary printed by
step 3 before trusting what you train.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!unzip -q -o /content/drive/MyDrive/leftover-lens.zip -d /content
%cd /content/leftover-lens

## 1. Install

In [ ]:
!pip -q install -e . ultralytics fiftyone
!python -c "import ultralytics; print('ultralytics', ultralytics.__version__)"

## 2. Convert Voxel51 to a YOLO segmentation dataset
Takes a few minutes (it downloads about 375 images).

In [ ]:
!python -m labeling.voxel51 --out /content/datasets/leftover

## 3. Train YOLO26n-seg
Runs as a separate process, so if Colab kills it (out of memory) you see `Killed` here and the notebook survives. Lighter settings than the config: `workers=0`, `batch=8`.

In [ ]:
%cd /content/leftover-lens
!free -h | head -2
!python -c "from vision.train import train; print(train(data='/content/datasets/leftover/data.yaml', project='/content/drive/MyDrive/leftover-runs', name='voxel51-v0', epochs=60, batch=8, workers=0, exist_ok=True))"

## 4. Export to ONNX

In [ ]:
%cd /content/leftover-lens
from pathlib import Path
best = Path('/content/drive/MyDrive/leftover-runs/voxel51-v0/weights/best.pt')
from vision.export_onnx import export_onnx
onnx_path = export_onnx(best)
print('onnx:', onnx_path)


## 5. Look at the predictions
Open a few validation images with the predicted masks drawn. Do the masks follow the food?

In [ ]:
from ultralytics import YOLO
import glob
model = YOLO(str(best))
files = sorted(glob.glob('/content/datasets/leftover/images/val/*.jpg'))[:6]
for r in model.predict(files, conf=0.25, save=True, project='/content/preds', name='val', exist_ok=True):
    pass
from IPython.display import Image, display
for f in sorted(glob.glob('/content/preds/val/*.jpg')):
    display(Image(f, width=480))

Everything is saved in `My Drive/leftover-runs/voxel51-v0/` (checkpoint, metrics, curves).
Send back: the class summary from step 2, the final epoch's metrics row, and a screenshot of two predictions.